# Module 04 - Tokenization

Use this notebook as the working space for the Module 04 exercises. Keep `notebooks/clean/` pristine; work in the copy created under `notebooks/solutions/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt

from g2c.tokenizer import BPETokenizer

## Before the Notebook

Use the tests to implement the tokenizer pieces first. The notebook assumes `_get_pair_counts`, `_merge`, `train_step`, `encode`, and `decode` are available as you progress. The scaffolded `train()` loop, Rust-backed `train_fast()`/`encode_fast()`, and tokenizer save/load helpers are implemented for you.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_tokenizer.py -x"
"Question: Which tokenizer test is the next one failing, and what implementation does it point at?"
"Answer: "

## Exercise 1 - Pair Counts and Merge

These two helpers are the BPE algorithm's core data-structure operations. Predict the outputs before running your implementation.

In [ ]:
"Question: What adjacent-pair counts should [1, 2, 1, 2, 3] produce?"
"Answer: "

"Question: Why does a list of length n have only n - 1 adjacent pairs?"
"Answer: "

"Question: What should _merge([1, 1, 1], (1, 1), 99) return, and why?"
"Answer: "

In [ ]:
pair_count_example = [1, 2, 1, 2, 3]
merge_example = [1, 1, 1]

print(BPETokenizer._get_pair_counts(pair_count_example))
print(BPETokenizer._merge(merge_example, (1, 1), 99))

## Exercise 2 - Train BPE on a Tiny Corpus

Train on a very small repeated string first. Call one `train_step()` by hand, then let scaffolded `train()` repeat that step until the target vocabulary size is reached. This makes the learned merges easy to inspect by hand.

In [ ]:
tiny_corpus = "the the the"

"Question: In the initial byte sequence for 'the the the', which adjacent pair should be most frequent?"
"Answer: "

"Question: If the target vocab size is 259, how many merges should be learned from a fresh tokenizer?"
"Answer: "

"Question: Why should the new token IDs start at 256?"
"Answer: "

In [ ]:
step_tok = BPETokenizer()
step_ids = list(tiny_corpus.encode("utf-8"))

print("one step:", step_tok.train_step(step_ids, new_id=len(step_tok.vocab)))
print("merges after one step:", step_tok.merges)
print("learned vocab after one step:", {i: step_tok.vocab[i] for i in range(256, len(step_tok.vocab))})


def print_bpe_progress(info: dict) -> None:
    print(
        f"step {info['steps']} | "
        f"vocab {info['vocab_size']}/{info['target_vocab_size']} | "
        f"last token {info['last_token_repr']} | "
        f"tokens {info['tokens']} | "
        f"last merge count {info['last_merge_count']} | "
        f"elapsed {info['elapsed_seconds']:.3f}s"
    )


tiny_tok = BPETokenizer()
tiny_tok.train(tiny_corpus, vocab_size=259, progress_callback=print_bpe_progress, progress_every=1)
print("merges:", tiny_tok.merges)
print("learned vocab:", {i: tiny_tok.vocab[i] for i in range(256, len(tiny_tok.vocab))})
assert len(tiny_tok.vocab) == 259

## Exercise 3 - Encode, Decode, and Round Trip

A byte-level tokenizer should round-trip any UTF-8 text, including text with characters that were not in the training corpus.

In [ ]:
"Question: Why can a byte-level tokenizer encode text it never saw during training?"
"Answer: "

"Question: During encode, why should the learned merge with the lowest new ID get priority?"
"Answer: "

"Question: What property of vocab entries makes decode lossless?"
"Answer: "

In [ ]:
roundtrip_texts = [
    "the theater there",
    "héllo wörld",
    "unseen text 12345 🌍",
]

roundtrip_tok = BPETokenizer()
roundtrip_tok.train("the quick brown fox jumps over the lazy dog " * 20, vocab_size=300)
for text in roundtrip_texts:
    ids = roundtrip_tok.encode(text)
    decoded = roundtrip_tok.decode(ids)
    print(text, "->", ids, "->", decoded)
    assert decoded == text

## Exercise 4 - Vocab Size vs Compression

Train several tokenizers at different vocab sizes and compare how many tokens they use for the same passage. The default corpus below is small for quick iteration; replace it with a larger text file for the full experiment.

In [ ]:
def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "g2c").exists():
            return candidate
    return path


def load_tokenizer_corpus(path: str | Path | None = None) -> str:
    """Load a text corpus, or use a small built-in corpus for smoke runs."""
    if path is not None:
        return Path(path).read_text(encoding="utf-8")

    paragraph = """
    The quick brown fox jumps over the lazy dog. The theater is there,
    and the theory of tokenization is that frequent text fragments become
    shorter symbols. In a small corpus, common words repeat; in a large corpus,
    BPE discovers useful subwords like ing, tion, the, and spaces before words.
    """
    return "\n".join(line.strip() for line in paragraph.splitlines() if line.strip()) * 40


corpus = load_tokenizer_corpus()
passage = corpus[:300]
print("corpus characters:", len(corpus))
print("passage characters:", len(passage))

In [ ]:
"Question: What do you predict will happen to token count as vocab size increases?"
"Answer: "

"Question: Why might the improvement show diminishing returns?"
"Answer: "

"Question: What is the tradeoff of using a very large vocabulary in a real model?"
"Answer: "

In [ ]:
def train_and_measure(corpus: str, passage: str, vocab_sizes: list[int]) -> list[dict]:
    """Train tokenizers and return compression statistics.

    Each result row should contain:
      - vocab_size_requested
      - vocab_size_actual
      - token_count
      - byte_count
      - chars_per_token
      - tokenizer
    """
    rows = []
    for vocab_size in vocab_sizes:
        # TODO: create BPETokenizer()
        # TODO: train it on corpus at vocab_size
        # TODO: encode passage
        # TODO: append a result dict with the fields listed above
        raise NotImplementedError("measure compression at each vocab size")
    return rows

In [ ]:
# Use small vocab sizes for the built-in corpus. For a larger corpus, try [256, 1024, 8192].
vocab_sizes = [256, 300, 400]

compression_rows = train_and_measure(corpus, passage, vocab_sizes)
for row in compression_rows:
    printable = {k: v for k, v in row.items() if k != "tokenizer"}
    print(printable)

plt.plot(
    [row["vocab_size_actual"] for row in compression_rows],
    [row["token_count"] for row in compression_rows],
    marker="o",
)
plt.xlabel("actual vocab size")
plt.ylabel("tokens in fixed passage")
plt.title("BPE compression improves as vocab grows")
plt.show()

## Exercise 5 - Inspect Learned Tokens

Print early and late learned vocab entries. Early learned IDs should usually be short, high-frequency byte patterns. Later learned IDs tend to be longer and more corpus-specific.

In [ ]:
"Question: Why should token IDs just above 255 usually represent very common patterns?"
"Answer: "

"Question: What kind of learned token would count as corpus-specific rather than generally useful?"
"Answer: "

In [ ]:
def token_bytes_display(token_bytes: bytes) -> str:
    """Display bytes as text when possible, with escapes for non-printing bytes."""
    return token_bytes.decode("utf-8", errors="backslashreplace")


def learned_vocab_window(tok: BPETokenizer, n: int = 20) -> tuple[list[tuple[int, str]], list[tuple[int, str]]]:
    """Return first and last n learned vocab entries as printable pairs."""
    # TODO: collect learned IDs, i.e. IDs >= 256
    # TODO: return first n and last n entries as (id, display_text) tuples
    raise NotImplementedError("inspect learned vocab entries")

In [ ]:
largest_tok = compression_rows[-1]["tokenizer"]
first_tokens, last_tokens = learned_vocab_window(largest_tok, n=20)

print("First learned tokens:")
for token_id, text in first_tokens:
    print(token_id, repr(text))

print("\nLast learned tokens:")
for token_id, text in last_tokens:
    print(token_id, repr(text))

In [ ]:
"Question: Name two early learned tokens and explain why they are plausible frequent patterns."
"Answer: "

"Question: Name one late learned token and explain what makes it more specific."
"Answer: "

## Exercise 6 - Optional Pre-Tokenization

GPT-2-style tokenizers split text before BPE so merges do not freely cross every boundary. This is optional; treat it as an extension after the core tokenizer passes tests.

In [ ]:
"Question: Why might merging across whitespace create less useful tokens?"
"Answer: "

"Question: What would you compare before and after adding pre-tokenization?"
"Answer: "

## Mini Milestone - Train Reusable Tokenizer Artifacts

The exercises above teach the BPE algorithm. This section turns that implementation into reusable course artifacts: trained tokenizers, encoded token streams, and inspection views that make the learned tokenization visible. Run this after the tokenizer tests pass.

Artifacts are written under `artifacts/tokenizers/`. Later modules can load the saved tokenizer JSON and token ID stream instead of retraining BPE from scratch.

In [ ]:
from array import array
from collections import Counter
from datetime import UTC, datetime
import gzip
import json
import random
import time

from IPython.display import Markdown, display

repo_root = find_repo_root()
tokenizer_artifact_root = repo_root / "artifacts" / "tokenizers"
end_of_text = "<|endoftext|>"
print("tokenizer artifacts:", tokenizer_artifact_root.relative_to(repo_root))

In [ ]:
def read_text_prefix(path: Path, max_chars: int) -> str:
    """Read up to max_chars from a UTF-8 text or gzip text file."""
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8", errors="replace") as f:
        return f.read(max_chars)


def load_tinyshakespeare_for_tokenizer(max_chars: int) -> str | None:
    path = repo_root / "data" / "tinyshakespeare.txt"
    if path.exists():
        return read_text_prefix(path, max_chars)
    return None


def load_tinystories_for_tokenizer(max_chars: int) -> str | None:
    candidates = [
        repo_root / "data" / "tinystories" / "TinyStories-train-100MB.txt",
        repo_root / "data" / "tinystories" / "TinyStories-train.txt",
    ]
    for path in candidates:
        if path.exists():
            return read_text_prefix(path, max_chars)
    return None


def load_g2c_corpus_for_tokenizer(corpus_dir: Path, max_chars: int) -> str | None:
    manifest_path = corpus_dir / "manifest.json"
    if not manifest_path.exists():
        return None

    manifest = json.loads(manifest_path.read_text())
    train_shards: list[Path] = []
    for source in manifest.get("sources", []):
        for shard in source.get("shards", []):
            if shard.get("split") == "train":
                train_shards.append(corpus_dir / shard["path"])

    parts: list[str] = []
    remaining = max_chars
    for shard_path in train_shards:
        if remaining <= 0:
            break
        chunk = read_text_prefix(shard_path, remaining)
        parts.append(chunk)
        remaining -= len(chunk)

    return "".join(parts) if parts else None


def load_artifact_source_text(source: str, max_chars: int) -> str | None:
    if source == "tinyshakespeare":
        return load_tinyshakespeare_for_tokenizer(max_chars)
    if source == "tinystories":
        return load_tinystories_for_tokenizer(max_chars)
    if source == "g2c-corpus-small":
        return load_g2c_corpus_for_tokenizer(repo_root / "data" / "g2c-corpus-v1-small", max_chars)
    if source == "g2c-corpus-full":
        return load_g2c_corpus_for_tokenizer(repo_root / "data" / "g2c-corpus-v1", max_chars)
    raise ValueError(f"unknown tokenizer source: {source}")

In [ ]:
def tokenizer_progress_every(vocab_size: int, updates: int = 80) -> int:
    return max(1, (vocab_size - 256) // updates)


def make_artifact_progress(label: str):
    start = time.perf_counter()
    progress = display(Markdown(f"{label} tokenizer training starting..."), display_id=True)

    def update(info: dict) -> None:
        last_token = info.get("last_token_repr")
        if last_token is None:
            last_token = "None"
        last_token = str(last_token).replace("`", "\\`")
        elapsed = info.get("elapsed_seconds", time.perf_counter() - start)
        progress.update(
            Markdown(
                f"{label}: vocab `{info['vocab_size']:,}/{info['target_vocab_size']:,}` "
                f"| steps `{info['steps']:,}` "
                f"| tokens `{info['tokens']:,}` "
                f"| last token `{last_token}` "
                f"| last count `{info['last_merge_count']}` "
                f"| elapsed `{elapsed:.1f}s`"
            )
        )

    return update


def save_token_ids(ids: list[int], path: Path) -> None:
    values = array("I", ids)
    with path.open("wb") as f:
        values.tofile(f)


def load_token_ids(path: Path) -> list[int]:
    values = array("I")
    with path.open("rb") as f:
        values.fromfile(f, path.stat().st_size // values.itemsize)
    return list(values)


def load_tokenizer_artifact(name: str) -> tuple[BPETokenizer, list[int], dict]:
    artifact_dir = tokenizer_artifact_root / name
    tokenizer = BPETokenizer.load(artifact_dir / "tokenizer.json")
    ids = load_token_ids(artifact_dir / "ids.uint32")
    manifest = json.loads((artifact_dir / "manifest.json").read_text())
    return tokenizer, ids, manifest


def train_or_load_tokenizer_artifact(config: dict, *, force: bool = False) -> dict | None:
    name = config["name"]
    artifact_dir = tokenizer_artifact_root / name
    tokenizer_path = artifact_dir / "tokenizer.json"
    ids_path = artifact_dir / "ids.uint32"
    manifest_path = artifact_dir / "manifest.json"

    if tokenizer_path.exists() and ids_path.exists() and manifest_path.exists() and not force:
        tokenizer, ids, manifest = load_tokenizer_artifact(name)
        print(f"loaded {name}: {len(ids):,} token IDs from {artifact_dir.relative_to(repo_root)}")
        text = load_artifact_source_text(config["source"], config["max_chars"])
        return {"config": config, "tokenizer": tokenizer, "ids": ids, "manifest": manifest, "text": text}

    text = load_artifact_source_text(config["source"], config["max_chars"])
    if not text:
        print(f"Skipping {name}: source {config['source']!r} is not available.")
        return None

    tokenizer = BPETokenizer()
    start = time.perf_counter()
    use_fast = config.get("use_fast", True)
    if config["vocab_size"] == 256:
        ids = list(text.encode("utf-8"))
    elif use_fast:
        progress = display(Markdown(f"{name}: Rust-backed tokenizer training starting..."), display_id=True)
        ids = tokenizer.train_fast(text, vocab_size=config["vocab_size"], show_progress=True)
        progress.update(
            Markdown(
                f"{name}: trained vocab `{len(tokenizer.vocab):,}/{config['vocab_size']:,}` "
                f"| tokens `{len(ids):,}` "
                f"| elapsed `{time.perf_counter() - start:.1f}s`"
            )
        )
    else:
        maybe_ids = tokenizer.train(
            text,
            vocab_size=config["vocab_size"],
            progress_callback=make_artifact_progress(name),
            progress_every=tokenizer_progress_every(config["vocab_size"]),
        )
        ids = maybe_ids if maybe_ids is not None else tokenizer.encode_fast(text)

    artifact_dir.mkdir(parents=True, exist_ok=True)
    tokenizer.save(tokenizer_path)
    save_token_ids(ids, ids_path)
    manifest = {
        "name": name,
        "kind": "bpe-tokenizer-artifact",
        "created_at": datetime.now(UTC).isoformat(),
        "source": config["source"],
        "requested_vocab_size": config["vocab_size"],
        "actual_vocab_size": len(tokenizer.vocab),
        "max_chars": config["max_chars"],
        "actual_chars": len(text),
        "token_count": len(ids),
        "chars_per_token": len(text) / max(1, len(ids)),
        "token_ids_file": "ids.uint32",
        "token_ids_dtype": "uint32",
        "trainer": "rust-tokenizers" if use_fast else "python-scaffold",
        "elapsed_seconds": time.perf_counter() - start,
        "notes": config.get("notes", ""),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"saved {name}: {len(ids):,} token IDs -> {artifact_dir.relative_to(repo_root)}")
    return {"config": config, "tokenizer": tokenizer, "ids": ids, "manifest": manifest, "text": text}


In [ ]:
RUN_OPTIONAL_TOKENIZER_ARTIFACTS = False
FORCE_RETRAIN_TOKENIZERS = False

TOKENIZER_ARTIFACT_CONFIGS = [
    {
        "name": "ShakespeareTokenizer",
        "source": "tinyshakespeare",
        "vocab_size": 2048,
        "max_chars": 1_000_000,
        "enabled": True,
        "notes": "Small reusable tokenizer for ShakespeareLM smoke and scaling runs.",
    },
    {
        "name": "StoryTokenizer",
        "source": "tinystories",
        "vocab_size": 4096,
        "max_chars": 5_000_000,
        "enabled": RUN_OPTIONAL_TOKENIZER_ARTIFACTS,
        "notes": "TinyStories tokenizer for StoryLM. Increase max_chars when you want the durable artifact.",
    },
    {
        "name": "G2CTokenizer-Small",
        "source": "g2c-corpus-small",
        "vocab_size": 8192,
        "max_chars": 5_000_000,
        "enabled": RUN_OPTIONAL_TOKENIZER_ARTIFACTS,
        "notes": "Broad course-corpus tokenizer for TinyLLM-small experiments.",
    },
]

for config in TOKENIZER_ARTIFACT_CONFIGS:
    status = "enabled" if config["enabled"] else "disabled"
    trainer = "fast" if config.get("use_fast", True) else "python"
    print(f"{config['name']}: {status}, trainer={trainer}, source={config['source']}, vocab={config['vocab_size']:,}, chars={config['max_chars']:,}")

print("Set RUN_OPTIONAL_TOKENIZER_ARTIFACTS = True after running ./datasets.sh --small or ./datasets.sh tinystories.")

In [ ]:
tokenizer_artifacts = []

for config in TOKENIZER_ARTIFACT_CONFIGS:
    if not config["enabled"]:
        print(f"Skipping {config['name']}: disabled in TOKENIZER_ARTIFACT_CONFIGS.")
        continue
    artifact = train_or_load_tokenizer_artifact(config, force=FORCE_RETRAIN_TOKENIZERS)
    if artifact is not None:
        tokenizer_artifacts.append(artifact)

print("trained/loaded artifacts:", [artifact["config"]["name"] for artifact in tokenizer_artifacts])

## Inspect What a Trained Tokenizer Learned

A BPE vocabulary is easiest to understand from several angles: a real tokenized passage, the learned tokens used most often on held text, and the longest learned tokens. Long tokens usually became possible because shorter pieces were frequent enough to merge repeatedly.

In [ ]:
def display_token_bytes(token_bytes: bytes) -> str:
    text = token_bytes.decode("utf-8", errors="backslashreplace")
    return text.replace("\n", "\\n").replace("\t", "\\t")


def shorten(text: str, width: int = 36) -> str:
    return text if len(text) <= width else text[: width - 1] + "…"


def deterministic_text_window(text: str, *, chars: int = 360, seed: int = 0) -> str:
    if len(text) <= chars:
        return text
    rng = random.Random(seed)
    start = rng.randrange(0, len(text) - chars)
    return text[start : start + chars]


def token_strings(tokenizer: BPETokenizer, text: str) -> list[str]:
    return [display_token_bytes(tokenizer.vocab[token_id]) for token_id in tokenizer.encode_fast(text)]


def most_frequent_learned_tokens(
    tokenizer: BPETokenizer,
    text: str,
    *,
    top_n: int = 20,
    max_chars: int = 200_000,
) -> list[tuple[int, str, int, int]]:
    ids = tokenizer.encode_fast(text[:max_chars])
    counts = Counter(token_id for token_id in ids if token_id >= 256)
    rows = []
    for token_id, count in counts.most_common(top_n):
        token_text = display_token_bytes(tokenizer.vocab[token_id])
        rows.append((token_id, token_text, count, len(tokenizer.vocab[token_id])))
    return rows


def longest_learned_tokens(tokenizer: BPETokenizer, *, top_n: int = 20) -> list[tuple[int, str, int]]:
    learned_ids = [token_id for token_id in tokenizer.vocab if token_id >= 256]
    learned_ids.sort(key=lambda token_id: (len(tokenizer.vocab[token_id]), token_id), reverse=True)
    return [
        (token_id, display_token_bytes(tokenizer.vocab[token_id]), len(tokenizer.vocab[token_id]))
        for token_id in learned_ids[:top_n]
    ]


def print_rows(title: str, headers: tuple[str, ...], rows: list[tuple]) -> None:
    print("\n" + title)
    print(" | ".join(headers))
    print("-" * 72)
    for row in rows:
        print(" | ".join(str(value) for value in row))


def plot_frequency_rows(rows: list[tuple[int, str, int, int]], *, title: str) -> None:
    if not rows:
        print("No learned tokens found in this sample.")
        return
    labels = [shorten(repr(token), 28) for _, token, _, _ in reversed(rows)]
    counts = [count for _, _, count, _ in reversed(rows)]
    plt.figure(figsize=(8, max(4, len(rows) * 0.28)))
    plt.barh(labels, counts)
    plt.xlabel("count in encoded sample")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
for artifact in tokenizer_artifacts:
    tokenizer = artifact["tokenizer"]
    text = artifact["text"]
    if not text:
        continue
    name = artifact["config"]["name"]
    manifest = artifact["manifest"]
    print("=" * 88)
    print(name)
    print("vocab size:", manifest["actual_vocab_size"])
    print("token count:", manifest["token_count"])
    print("chars/token:", round(manifest["chars_per_token"], 3))

    sample = deterministic_text_window(text, chars=360, seed=13)
    print("\nSample text:")
    print(repr(sample))
    print("\nToken strings for sample:")
    print(token_strings(tokenizer, sample))

    frequent_rows = most_frequent_learned_tokens(tokenizer, text, top_n=20)
    print_rows(
        "Most frequent learned tokens in sample",
        ("token_id", "token", "count", "bytes"),
        [(token_id, repr(shorten(token)), count, length) for token_id, token, count, length in frequent_rows],
    )

    longest_rows = longest_learned_tokens(tokenizer, top_n=20)
    print_rows(
        "Longest learned tokens",
        ("token_id", "token", "bytes"),
        [(token_id, repr(shorten(token)), length) for token_id, token, length in longest_rows],
    )

    plot_frequency_rows(frequent_rows, title=f"{name}: most frequent learned tokens")

In [ ]:
artifact_name = "ShakespeareTokenizer"
artifact_dir = tokenizer_artifact_root / artifact_name

if (artifact_dir / "tokenizer.json").exists():
    tokenizer, ids, manifest = load_tokenizer_artifact(artifact_name)
    print(manifest["actual_vocab_size"], len(ids))
else:
    print(f"Run the artifact training cell first to create {artifact_dir.relative_to(repo_root)}.")

## Submission Notes

When complete, ask a coding agent to grade your Module 04 notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.